# BPM Prediction - Improved Approach (No Overfitting)

This notebook fixes the overfitting issues by:
1. Using cross-validation for ensemble weight optimization
2. Creating diverse models without external predictions
3. Implementing proper validation-based model selection
4. Adding stacking as an alternative approach

In [1]:
# Import libraries (same as original)
import json
import os
import pickle
import time
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    root_mean_squared_error
)
from sklearn.model_selection import (
    train_test_split, GridSearchCV, KFold, cross_val_score,
    cross_val_predict
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.svm import SVR

import xgboost as xg
import lightgbm as lgb
import catboost

import optuna
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

print("Libraries successfully loaded. Ready to go!")

Libraries successfully loaded. Ready to go!


c:\Users\robkr\anaconda3\envs\ml-env-stable\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (524164, 11)
Test shape: (174722, 10)


In [3]:
# Feature engineering (same as original)
def create_features(df):
    """Create additional features that might help predict BPM"""
    df = df.copy()
    
    # 1. Rhythm and Energy interactions
    df['RhythmEnergyProduct'] = df['RhythmScore'] * df['Energy']
    df['RhythmEnergyRatio'] = df['RhythmScore'] / (df['Energy'] + 1e-8)
    
    # 2. Audio characteristics
    df['LoudnessEnergyProduct'] = df['AudioLoudness'] * df['Energy']
    df['VocalInstrumentalRatio'] = df['VocalContent'] / (df['InstrumentalScore'] + 1e-8)
    
    # 3. Track duration features
    df['TrackDurationMin'] = df['TrackDurationMs'] / 60000  # Convert to minutes
    df['DurationMoodProduct'] = df['TrackDurationMin'] * df['MoodScore']
    
    # 4. Performance and quality features
    df['QualityPerformanceProduct'] = df['AcousticQuality'] * df['LivePerformanceLikelihood'] * df['VocalContent']

    # 5. Polynomial features for key features
    key_features = ['MoodScore', 'TrackDurationMs', 'RhythmScore']
    for feature in key_features:
        df[f'{feature}_squared'] = df[feature] ** 2
        df[f'{feature}_sqrt'] = np.sqrt(np.abs(df[feature]))
    
    # 6. Additional interaction features
    df['RhythmDurationInteraction'] = df['RhythmScore'] * df['TrackDurationMin']
    df['QualityMoodProduct'] = df['AcousticQuality'] * df['MoodScore']
    df['LiveVocalInteraction'] = df['LivePerformanceLikelihood'] * df['VocalContent']
    df['EnergyPerMinute'] = df['Energy'] / df['TrackDurationMin']
    
    # 7. Logarithmic transformations
    df['LogDuration'] = np.log1p(df['TrackDurationMin'])
    df['LogEnergy'] = np.log1p(df['Energy'])
    df['LogLoudness'] = np.log1p(np.abs(df['AudioLoudness']) + 1)
    
    return df

# Apply feature engineering
train_engineered = create_features(train)
test_engineered = create_features(test)

print(f"Created {len([col for col in train_engineered.columns if col not in train.columns])} new features")

Created 20 new features


In [4]:
# Prepare data for modeling
target_col = 'BeatsPerMinute'
numerical_features = train_engineered.select_dtypes(include=[np.number]).columns
feature_columns = [col for col in numerical_features if col not in ['id', target_col]]

X = train_engineered[feature_columns]
y = train_engineered[target_col]
X_test = test_engineered[feature_columns]

print(f"Training with {len(feature_columns)} features")
print(f"Target variable: {target_col}")

Training with 29 features
Target variable: BeatsPerMinute


In [5]:
# Cross-validation based ensemble weight optimization
def find_optimal_ensemble_weights(models, X, y, cv_folds=5):
    """
    Find optimal ensemble weights using cross-validation.
    This avoids overfitting to any specific validation set.
    """
    kf = KFold(n_splits=cv_folds, shuffle=True, random_state=SEED)
    
    # Get cross-validated predictions for each model
    cv_predictions = {}
    
    for name, model in models.items():
        print(f"Getting CV predictions for {name}...")
        if name in ['Ridge', 'ElasticNet', 'SVR']:
            # Scale features for linear models
            scaler = RobustScaler()
            pipeline = make_pipeline(scaler, model)
            cv_pred = cross_val_predict(pipeline, X, y, cv=kf)
        else:
            cv_pred = cross_val_predict(model, X, y, cv=kf)
        
        cv_predictions[name] = cv_pred
    
    # Find optimal weights using grid search on CV predictions
    model_names = list(models.keys())
    n_models = len(model_names)
    
    best_score = float('inf')
    best_weights = None
    
    # Grid search over weight combinations
    weight_options = np.arange(0, 1.1, 0.1)
    
    if n_models == 2:
        for w1 in weight_options:
            w2 = 1 - w1
            weights = [w1, w2]
            
            ensemble_pred = sum(w * cv_predictions[model_names[i]] 
                              for i, w in enumerate(weights))
            
            score = root_mean_squared_error(y, ensemble_pred)
            
            if score < best_score:
                best_score = score
                best_weights = weights
    
    elif n_models == 3:
        for w1 in weight_options:
            for w2 in weight_options:
                w3 = 1 - w1 - w2
                if w3 >= 0 and w3 <= 1:
                    weights = [w1, w2, w3]
                    
                    ensemble_pred = sum(w * cv_predictions[model_names[i]] 
                                      for i, w in enumerate(weights))
                    
                    score = root_mean_squared_error(y, ensemble_pred)
                    
                    if score < best_score:
                        best_score = score
                        best_weights = weights
    
    return best_weights, best_score, cv_predictions

print("Function defined for CV-based weight optimization")

Function defined for CV-based weight optimization


In [6]:
# Define diverse models (no external predictions)
models = {
    'XGBoost': xg.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=2.0,
        random_state=SEED,
        n_jobs=-1
    ),
    'LightGBM': lgb.LGBMRegressor(
        n_estimators=400,
        learning_rate=0.08,
        max_depth=5,
        num_leaves=93,
        min_child_samples=50,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1
    ),
    'Ridge': Ridge(alpha=1.0)
}

print(f"Defined {len(models)} diverse models for ensemble")

Defined 3 diverse models for ensemble


In [7]:
# Find optimal ensemble weights using cross-validation
print("🔍 Finding optimal ensemble weights using cross-validation...")
optimal_weights, cv_score, cv_predictions = find_optimal_ensemble_weights(models, X, y)

print(f"\n🏆 Optimal ensemble weights:")
model_names = list(models.keys())
for i, (name, weight) in enumerate(zip(model_names, optimal_weights)):
    print(f"   {weight*100:5.1f}% {name}")

print(f"\n📊 Cross-validation RMSE: {cv_score:.4f}")

🔍 Finding optimal ensemble weights using cross-validation...
Getting CV predictions for XGBoost...
Getting CV predictions for LightGBM...
Getting CV predictions for Ridge...

🏆 Optimal ensemble weights:
    30.0% XGBoost
    10.0% LightGBM
    60.0% Ridge

📊 Cross-validation RMSE: 26.4631


In [13]:
# Train final models on full dataset and make predictions
print("🚀 Training final models on full dataset...")

final_predictions = np.zeros(len(X_test))
scaler = RobustScaler()

for i, (name, model) in enumerate(models.items()):
    print(f"   Training {name}...")
    
    if name in ['Ridge', 'ElasticNet', 'SVR']:
        # Scale features for linear models
        X_scaled = scaler.fit_transform(X)
        X_test_scaled = scaler.transform(X_test)
        
        model.fit(X_scaled, y)
        pred = model.predict(X_test_scaled)
    else:
        model.fit(X, y)
        pred = model.predict(X_test)
    
    weight = optimal_weights[i]
    final_predictions += weight * pred
    
    print(f"     Weight: {weight:.3f}, Prediction range: {pred.min():.2f} - {pred.max():.2f}")

print(f"\n🎯 Final ensemble prediction range: {final_predictions.min():.2f} - {final_predictions.max():.2f}")

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test['id'],
    'BeatsPerMinute': final_predictions
})

🚀 Training final models on full dataset...
   Training XGBoost...
     Weight: 0.300, Prediction range: 106.86 - 130.11
   Training LightGBM...
     Weight: 0.100, Prediction range: 105.37 - 134.09
   Training Ridge...
     Weight: 0.600, Prediction range: 103.09 - 121.27

🎯 Final ensemble prediction range: 108.40 - 124.27


In [14]:
submission.to_csv("submission_c.csv", index=False)
submission

,id,BeatsPerMinute
0,524164,118.892254
1,524165,118.776762
2,524166,119.678554
3,524167,119.609328
4,524168,119.617345
...,...,...
174717,698881,118.732387
174718,698882,119.721863
174719,698883,118.939945
174720,698884,118.752776


In [9]:
# Alternative: Stacking approach
print("🏗️ Training stacking ensemble as alternative...")

# Create base models for stacking
base_models = [
    ('xgb', xg.XGBRegressor(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        random_state=SEED, n_jobs=-1
    )),
    ('lgb', lgb.LGBMRegressor(
        n_estimators=200, learning_rate=0.08, max_depth=4,
        random_state=SEED, n_jobs=-1, verbose=-1
    )),
    ('ridge', make_pipeline(RobustScaler(), Ridge(alpha=1.0)))
]

# Meta-learner
meta_learner = LinearRegression()

# Create stacking regressor
stacking_regressor = StackingRegressor(
    estimators=base_models,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)

# Train stacking model
stacking_regressor.fit(X, y)
stacking_predictions = stacking_regressor.predict(X_test)

# Evaluate stacking with cross-validation
stacking_cv_scores = cross_val_score(
    stacking_regressor, X, y, cv=5,
    scoring='neg_root_mean_squared_error', n_jobs=-1
)
stacking_cv_score = -stacking_cv_scores.mean()

print(f"📊 Stacking CV RMSE: {stacking_cv_score:.4f} ± {stacking_cv_scores.std():.4f}")
print(f"🎯 Stacking prediction range: {stacking_predictions.min():.2f} - {stacking_predictions.max():.2f}")

🏗️ Training stacking ensemble as alternative...
📊 Stacking CV RMSE: 26.4621 ± 0.0489
🎯 Stacking prediction range: 114.49 - 125.09


In [10]:
# Choose best approach based on CV scores
print("\n🏆 COMPARISON OF APPROACHES:")
print(f"   Weighted Ensemble CV RMSE: {cv_score:.4f}")
print(f"   Stacking CV RMSE: {stacking_cv_score:.4f}")

if cv_score < stacking_cv_score:
    chosen_predictions = final_predictions
    chosen_method = "Weighted Ensemble"
    print(f"\n✅ Using {chosen_method} (better CV score)")
else:
    chosen_predictions = stacking_predictions
    chosen_method = "Stacking"
    print(f"\n✅ Using {chosen_method} (better CV score)")


🏆 COMPARISON OF APPROACHES:
   Weighted Ensemble CV RMSE: 26.4631
   Stacking CV RMSE: 26.4621

✅ Using Stacking (better CV score)


In [11]:
# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'BeatsPerMinute': chosen_predictions
})

# Save submission
submission.to_csv('submission_improved.csv', index=False)

print(f"✅ Submission saved as 'submission_improved.csv'")
print(f"🎯 Final BPM range: {chosen_predictions.min():.2f} - {chosen_predictions.max():.2f}")
print(f"📊 Method used: {chosen_method}")

# Display first few predictions
print("\n📋 First 10 predictions:")
print(submission.head(10))

✅ Submission saved as 'submission_improved.csv'
🎯 Final BPM range: 114.49 - 125.09
📊 Method used: Stacking

📋 First 10 predictions:
       id  BeatsPerMinute
0  524164      119.193538
1  524165      118.701766
2  524166      119.739746
3  524167      119.360270
4  524168      119.593174
5  524169      119.178984
6  524170      119.017045
7  524171      118.238003
8  524172      119.137256
9  524173      118.408200


In [12]:
# Validation: Compare individual model performance
print("\n🔍 INDIVIDUAL MODEL CROSS-VALIDATION SCORES:")
print("="*50)

for name, model in models.items():
    if name in ['Ridge', 'ElasticNet', 'SVR']:
        pipeline = make_pipeline(RobustScaler(), model)
        cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')
    else:
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    
    cv_rmse = -cv_scores.mean()
    cv_std = cv_scores.std()
    
    print(f"{name:<12} | RMSE: {cv_rmse:.4f} ± {cv_std:.4f}")

print(f"\nEnsemble      | RMSE: {cv_score:.4f} (weighted)")
print(f"Stacking      | RMSE: {stacking_cv_score:.4f}")


🔍 INDIVIDUAL MODEL CROSS-VALIDATION SCORES:
XGBoost      | RMSE: 26.4707 ± 0.0510
LightGBM     | RMSE: 26.4888 ± 0.0511
Ridge        | RMSE: 26.4658 ± 0.0486

Ensemble      | RMSE: 26.4631 (weighted)
Stacking      | RMSE: 26.4621
